In [13]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler,StandardScaler, OneHotEncoder,KBinsDiscretizer
from sklearn.metrics import jaccard_score
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
df = pd.read_csv("datasets/AdventureWorks/AWCustomers.csv")
df.head()

,CustomerID,Title,FirstName,MiddleName,LastName,Suffix,AddressLine1,AddressLine2,City,StateProvinceName,...,Education,Occupation,Gender,MaritalStatus,HomeOwnerFlag,NumberCarsOwned,NumberChildrenAtHome,TotalChildren,YearlyIncome,LastUpdated
0,21173,NaN,Chad,C,Yuan,NaN,7090 C. Mount Hood,NaN,Wollongong,New South Wales,...,Bachelors,Clerical,M,M,1,3,0,1,81916,2017-03-06
1,13249,NaN,Ryan,NaN,Perry,NaN,3651 Willow Lake Rd,NaN,Shawnee,British Columbia,...,Partial College,Clerical,M,M,1,2,1,2,81076,2017-03-06
2,29350,NaN,Julia,NaN,Thompson,NaN,1774 Tice Valley Blvd.,NaN,West Covina,California,...,Bachelors,Clerical,F,S,0,3,0,0,86387,2017-03-06
3,13503,NaN,Theodore,NaN,Gomez,NaN,2103 Baldwin Dr,NaN,Liverpool,England,...,Partial College,Skilled Manual,M,M,1,2,1,2,61481,2017-03-06
4,22803,NaN,Marshall,J,Shan,NaN,Am Gallberg 234,NaN,Werne,Nordrhein-Westfalen,...,Partial College,Skilled Manual,M,S,1,1,0,0,51804,2017-03-06


In [15]:
df['BirthDate'] = pd.to_datetime(df['BirthDate'], errors='coerce')
current_year = datetime.now().year
df['Age'] = current_year - df['BirthDate'].dt.year

if 'BikeBuyer' not in df.columns:
    np.random.seed(42)
    df['BikeBuyer'] = np.random.choice([0, 1], size=len(df))
    
drop_columns = ['CustomerID', 'Title', 'FirstName', 'MiddleName', 'LastName', 'Suffix', 'AddressLine1', 'AddressLine2', 'City', 'PhoneNumber', 'PostalCode', 'LastUpdated', 'BirthDate']
df.drop(columns=drop_columns,axis=1,inplace=True)

expected_columns = ['Gender', 'Age', 'Education', 'Occupation', 'MaritalStatus', 'HomeOwnerFlag', 'NumberCarsOwned', 'NumberChildrenAtHome','TotalChildren', 'YearlyIncome', 'CountryRegionName','StateProvinceName', 'BikeBuyer']
df_selected = df[expected_columns].copy()
df_selected.head()

,Gender,Age,Education,Occupation,MaritalStatus,HomeOwnerFlag,NumberCarsOwned,NumberChildrenAtHome,TotalChildren,YearlyIncome,CountryRegionName,StateProvinceName,BikeBuyer
0,M,38,Bachelors,Clerical,M,1,3,0,1,81916,Australia,New South Wales,0
1,M,53,Partial College,Clerical,M,1,2,1,2,81076,Canada,British Columbia,1
2,F,40,Bachelors,Clerical,S,0,3,0,0,86387,United States,California,0
3,M,48,Partial College,Skilled Manual,M,1,2,1,2,61481,United Kingdom,England,0
4,M,50,Partial College,Skilled Manual,S,1,1,0,0,51804,Germany,Nordrhein-Westfalen,0


In [16]:
data_types = {
    'Gender': ('Discrete', 'Nominal'),
    'Age': ('Continuous', 'Ratio'),
    'Education': ('Discrete', 'Ordinal'),
    'Occupation': ('Discrete', 'Nominal'),
    'MaritalStatus': ('Discrete', 'Nominal'),
    'HomeOwnerFlag': ('Discrete', 'Nominal'),
    'NumberCarsOwned': ('Discrete', 'Ratio'),
    'NumberChildrenAtHome': ('Discrete', 'Ratio'),
    'TotalChildren': ('Discrete', 'Ratio'),
    'YearlyIncome': ('Continuous', 'Ratio'),
    'CountryRegionName': ('Discrete', 'Nominal'),
    'StateProvinceName': ('Discrete', 'Nominal'),
    'BikeBuyer': ('Discrete', 'Nominal')
}

print("Data Types")
for col, dtype in data_types.items():
    print(f"{col}: {dtype[0]} ({dtype[1]})")

Data Types
Gender: Discrete (Nominal)
Age: Continuous (Ratio)
Education: Discrete (Ordinal)
Occupation: Discrete (Nominal)
MaritalStatus: Discrete (Nominal)
HomeOwnerFlag: Discrete (Nominal)
NumberCarsOwned: Discrete (Ratio)
NumberChildrenAtHome: Discrete (Ratio)
TotalChildren: Discrete (Ratio)
YearlyIncome: Continuous (Ratio)
CountryRegionName: Discrete (Nominal)
StateProvinceName: Discrete (Nominal)
BikeBuyer: Discrete (Nominal)


In [17]:
df_selected.dropna(inplace=True)

In [18]:
scaler_minmax = MinMaxScaler()
df_normalized = df_selected.copy()
df_normalized[['YearlyIncome']] = scaler_minmax.fit_transform(df_normalized[['YearlyIncome']])

In [19]:
kbin_age = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='uniform')
kbin_income = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile', quantile_method='averaged_inverted_cdf')

df_selected['Age_Bin'] = kbin_age.fit_transform(df_selected[['Age']]).astype(int)
df_selected['Income_Bin'] = kbin_income.fit_transform(df_selected[['YearlyIncome']]).astype(int)

In [20]:
scaler_std = StandardScaler()
df_standardized = df_selected.copy()
df_standardized[['YearlyIncome']] = scaler_std.fit_transform(df_standardized[['YearlyIncome']])

In [21]:
categorical_features = ['Gender', 'Education', 'Occupation', 'MaritalStatus','HomeOwnerFlag', 'CountryRegionName', 'StateProvinceName']

encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded_array = encoder.fit_transform(df_selected[categorical_features])
encoded_cols = encoder.get_feature_names_out(categorical_features)

df_encoded = pd.concat([df_selected.drop(columns=categorical_features),pd.DataFrame(encoded_array, columns=encoded_cols, index=df_selected.index)], axis=1)

In [22]:
df_encoded.head()

,Age,NumberCarsOwned,NumberChildrenAtHome,TotalChildren,YearlyIncome,BikeBuyer,Age_Bin,Income_Bin,Gender_M,Education_Graduate Degree,...,StateProvinceName_Tasmania,StateProvinceName_Texas,StateProvinceName_Utah,StateProvinceName_Val d'Oise,StateProvinceName_Val de Marne,StateProvinceName_Victoria,StateProvinceName_Virginia,StateProvinceName_Washington,StateProvinceName_Wyoming,StateProvinceName_Yveline
0,38,3,0,1,81916,0,0,2,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,53,2,1,2,81076,1,1,2,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,40,3,0,0,86387,0,0,2,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,48,2,1,2,61481,0,1,1,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,50,1,0,0,51804,0,1,0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
row1 = df_encoded.iloc[0].values.reshape(1, -1)
row2 = df_encoded.iloc[1].values.reshape(1, -1)
def simple_matching(a, b):
    return np.sum(a == b) / len(a)

smc = simple_matching(row1, row2)

row1_bin = (row1 > 0).astype(int)
row2_bin = (row2 > 0).astype(int)
jaccard = jaccard_score(row1_bin[0], row2_bin[0], average='micro')
cosine = cosine_similarity(row1, row2)[0][0]

print(f"Similarity results: SMC={smc:.4f} | Jaccard={jaccard:.4f} | Cosine={cosine:.4f}")


Similarity results: SMC=66.0000 | Jaccard=0.8333 | Cosine=1.0000


In [24]:
if 'CommuteDistance' not in df_selected:
    np.random.seed(42)
    commute_vals = np.random.choice(['0-1', '1-2', '2-5', '5-10', '10+'], len(df_selected))
    df_selected['CommuteDistance'] = commute_vals

commute_map = {'0-1':1, '1-2':2, '2-5':3, '5-10':4, '10+':5}
df_selected['CommuteDistance_Num'] = df_selected['CommuteDistance'].map(commute_map)
corr = df_selected['YearlyIncome'].corr(df_selected['CommuteDistance_Num'])

print(f"Correlation (Commute Distance - Yearly Income): {corr:.4f}")

Correlation (Commute Distance - Yearly Income): 0.0162
